<a href="https://colab.research.google.com/github/Koshalendar/Artificial-intelligence-/blob/main/resolution_using_FOL__AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

class Node:
    def __init__(self, clause, left=None, right=None, subst=None):
        self.clause = clause
        self.left = left
        self.right = right
        self.subst = subst or {}

def parse_literal(lit):
    lit = lit.strip()
    neg = lit.startswith("~")
    if neg:
        lit = lit[1:].strip()
    match = re.match(r"([A-Za-z0-9_]+)\(([^)]*)\)", lit)
    if not match:
        raise ValueError(f"Invalid literal format: '{lit}'. Expected Predicate(arg1, arg2).")
    pred = match.group(1)
    args = [a.strip() for a in match.group(2).split(",")]
    return (neg, pred, args)

def format_literal(lit_tuple):
    neg, pred, args = lit_tuple
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "[]  <-- EMPTY CLAUSE (CONTRADICTION)"
    return " | ".join(format_literal(l) for l in clause)

def is_variable(term):
    return term[0].islower()

def unify_terms(t1, t2, subst):
    t1 = subst.get(t1, t1)
    t2 = subst.get(t2, t2)
    if t1 == t2:
        return subst
    if is_variable(t1):
        new_subst = subst.copy()
        new_subst[t1] = t2
        return new_subst
    if is_variable(t2):
        new_subst = subst.copy()
        new_subst[t2] = t1
        return new_subst
    return None

def unify_literals(l1, l2):
    neg1, pred1, args1 = l1
    neg2, pred2, args2 = l2
    if neg1 == neg2 or pred1 != pred2 or len(args1) != len(args2):
        return None
    subst = {}
    for a1, a2 in zip(args1, args2):
        subst = unify_terms(a1, a2, subst)
        if subst is None:
            return None
    return subst

def apply_subst(clause, subst):
    result = []
    for neg, pred, args in clause:
        new_args = [subst.get(a, a) for a in args]
        lit = (neg, pred, new_args)
        if lit not in result:
            result.append(lit)
    return result

def resolve(node1, node2):
    resolvents = []
    for i, l1 in enumerate(node1.clause):
        for j, l2 in enumerate(node2.clause):
            subst = unify_literals(l1, l2)
            if subst is not None:
                rem1 = [l for idx, l in enumerate(node1.clause) if idx != i]
                rem2 = [l for idx, l in enumerate(node2.clause) if idx != j]
                new_clause = apply_subst(rem1 + rem2, subst)
                resolvents.append(Node(new_clause, node1, node2, subst))
    return resolvents

def print_proof_tree(node, indent="", is_last=True):
    branch = "└── " if is_last else "├── "
    subst_str = f"   {{Substitution: {node.subst}}}" if node.subst else ""
    print(f"{indent}{branch}{format_clause(node.clause)}{subst_str}")

    children = [c for c in (node.left, node.right) if c is not None]
    for idx, child in enumerate(children):
        next_indent = indent + ("    " if is_last else "│   ")
        print_proof_tree(child, next_indent, idx == len(children) - 1)

def main():
    print("======================================================")
    print("   FIRST-ORDER LOGIC (FOL) RESOLUTION PROOF TREE")
    print("======================================================")
    print("Rules for entering clauses in CNF:")
    print("  1. Use '~' for NOT, '|' for OR")
    print("  2. Lowercase = variable (e.g., x, y)")
    print("  3. Uppercase = constant/predicate (e.g., John, Peanuts, Food)")
    print("  Example: ~Food(x) | Likes(John, x)\n")

    n = int(input("Enter the total number of CNF premises in Knowledge Base: "))
    nodes = []

    print("\n--- Enter Premises ---")
    for i in range(1, n + 1):
        while True:
            try:
                clause_str = input(f"Premise {i}: ").strip()
                lits = [parse_literal(l) for l in clause_str.split("|")]
                nodes.append(Node(lits))
                break
            except Exception as e:
                print(f"Error: {e}. Please try again.")

    print("\n--- Enter Goal ---")
    while True:
        try:
            goal_str = input("Enter the goal to prove (e.g., Likes(John, Peanuts)): ").strip()
            goal_lit = parse_literal(goal_str)
            # Negate the goal for resolution refutation
            negated_goal = [(not goal_lit[0], goal_lit[1], goal_lit[2])]
            nodes.append(Node(negated_goal))
            print(f"Negated Goal added to KB: {format_clause(negated_goal)}\n")
            break
        except Exception as e:
            print(f"Error: {e}. Please try again.")

    # Resolution Loop
    seen = {format_clause(node.clause) for node in nodes}
    proof_root = None

    while proof_root is None:
        new_nodes = []
        total = len(nodes)
        for i in range(total):
            for j in range(i + 1, total):
                for res in resolve(nodes[i], nodes[j]):
                    if len(res.clause) == 0:
                        proof_root = res
                        break
                    sig = format_clause(res.clause)
                    if sig not in seen:
                        seen.add(sig)
                        new_nodes.append(res)
                if proof_root:
                    break
            if proof_root:
                break
        if not new_nodes and proof_root is None:
            break
        nodes.extend(new_nodes)

    print("======================================================")
    print("               RESOLUTION PROOF TREE")
    print("======================================================")
    if proof_root:
        print_proof_tree(proof_root)
        print("\nCONCLUSION: Goal PROVED by contradiction!")
    else:
        print("CONCLUSION: Could not prove the goal from the given premises.")

if __name__ == "__main__":
    main()



   FIRST-ORDER LOGIC (FOL) RESOLUTION PROOF TREE
Rules for entering clauses in CNF:
  1. Use '~' for NOT, '|' for OR
  2. Lowercase = variable (e.g., x, y)
  3. Uppercase = constant/predicate (e.g., John, Peanuts, Food)
  Example: ~Food(x) | Likes(John, x)

Enter the total number of CNF premises in Knowledge Base: 9

--- Enter Premises ---
Premise 1:  ~Food(x) | Likes(John, x)
Premise 2:  Food(Apple)
Premise 3:  Food(Vegetables)
Premise 4: ~Eats(y, x) | Killed(y) | Food(x)
Premise 5:  Eats(Anil, Peanuts)
Premise 6:  Alive(Anil)
Premise 7:  ~Eats(Anil, x) | Eats(Harry, x) 
Premise 8:   ~Alive(x) | ~Killed(x)
Premise 9: Killed(x) | Alive(x)

--- Enter Goal ---
Enter the goal to prove (e.g., Likes(John, Peanuts)):  Likes(John, Peanuts)
Negated Goal added to KB: ~Likes(John, Peanuts)

               RESOLUTION PROOF TREE
└── []  <-- EMPTY CLAUSE (CONTRADICTION)
    ├── ~Food(Peanuts)   {Substitution: {'x': 'Peanuts'}}
    │   ├── ~Food(x) | Likes(John, x)
    │   └── ~Likes(John, Peanuts)
